# 01 – Data preparation

**Project:** Prediction of football transfer fees (Transfermarkt data).
**Target:** `transfer_fee` (modelled later as `log1p(transfer_fee)`).

This notebook covers:
1. Description of the datasets used for modelling (attributes, initial state, attributes selected for prediction).
2. Cleaning of missing values. Safe, unambiguous steps are done directly; every step that needs a decision is only *diagnosed* here and marked **PENDING DECISION**.

Only the 6 tables needed for modelling are described. The other tables in `data/raw` (games, game_events, game_lineups, club_games, countries, national_teams) are not used.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 50)
RAW = Path("data/raw")

transfers         = pd.read_csv(RAW / "transfers.csv")
players           = pd.read_csv(RAW / "players.csv")
player_valuations = pd.read_csv(RAW / "player_valuations.csv")
appearances       = pd.read_csv(RAW / "appearances.csv")
clubs             = pd.read_csv(RAW / "clubs.csv")
competitions      = pd.read_csv(RAW / "competitions.csv")

tables = {
    "transfers": transfers, "players": players, "player_valuations": player_valuations,
    "appearances": appearances, "clubs": clubs, "competitions": competitions,
}

## 1. Datasets and their initial state

Source: *Football Data from Transfermarkt* (Kaggle, `davidcariboo/player-scores`; snapshot downloaded in November 2026, source updates are currently paused).

| Table | Grain (one row =) | Role in the project |
|---|---|---|
| `transfers` | one transfer of a player between two clubs | **Base table**, contains the target `transfer_fee` |
| `players` | one player | Player profile (age, position, foot, height, citizenship) |
| `player_valuations` | one market-value record of a player at a date | Market value *before* the transfer, value trend |
| `appearances` | one player in one match | Performance in the period *before* the transfer (goals, assists, minutes, cards) |
| `clubs` | one club | Selling / buying club profile |
| `competitions` | one competition | League level and country of the clubs |

In [3]:
overview = pd.DataFrame({
    name: {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing cells %": round(df.isna().mean().mean() * 100, 2),
        "duplicate rows": int(df.duplicated().sum()),
    }
    for name, df in tables.items()
}).T
overview

,rows,columns,missing cells %,duplicate rows
transfers,175165.0,10.0,7.40,0.0
players,50149.0,26.0,15.29,0.0
player_valuations,656301.0,6.0,2.36,0.0
appearances,1894350.0,13.0,0.00,0.0
clubs,796.0,17.0,9.51,0.0
competitions,65.0,11.0,5.45,0.0


In [4]:
def missing_report(df):
    m = df.isna().sum()
    out = pd.DataFrame({
        "missing": m,
        "missing_%": (m / len(df) * 100).round(2),
        "dtype": df.dtypes.astype(str),
    })
    return out[out["missing"] > 0].sort_values("missing", ascending=False)

for name, df in tables.items():
    print(f"--- {name} {df.shape} ---")
    display(missing_report(df))

--- transfers (175165, 10) ---


,missing,missing_%,dtype
market_value_in_eur,68160,38.91,float64
transfer_fee,61526,35.12,float64


--- players (50149, 26) ---


,missing,missing_%,dtype
current_national_team_id,46889,93.50,float64
international_goals,30793,61.40,float64
international_caps,30793,61.40,float64
agent_name,23296,46.45,object
contract_expiration_date,18569,37.03,object
market_value_in_eur,8621,17.19,float64
highest_market_value_in_eur,8621,17.19,float64
foot,5893,11.75,object
country_of_birth,5885,11.74,object
city_of_birth,5608,11.18,object


--- player_valuations (656301, 6) ---


,missing,missing_%,dtype
player_club_domestic_competition_id,92973,14.17,object


... (2 outputs left out)

--- clubs (796, 17) ---


,missing,missing_%,dtype
total_market_value,796,100.00,float64
coach_name,393,49.37,object
foreigners_percentage,60,7.54,float64
average_age,38,4.77,float64


--- competitions (65, 11) ---


,missing,missing_%,dtype
total_clubs,15,23.08,float64
country_name,12,18.46,object
domestic_league_code,12,18.46,object


### Attributes selected for prediction

**`transfers`** (base table)
- `player_id`, `transfer_date`, `transfer_season`, `from_club_id`, `to_club_id` – keys, time and the clubs involved
- `from_club_name`, `to_club_name`, `player_name` – kept only for readability / error analysis, not model inputs
- `transfer_fee` – **target**
- `market_value_in_eur` – market value at the transfer date (strong feature; **missing for ~39 % of rows** – these rows are dropped, see section 2.2)

**`players`**
- `player_id`, `name`, `date_of_birth` (→ age at transfer), `position`, `sub_position`, `foot`, `height_in_cm`, `country_of_citizenship`
- Deliberately **not used** (snapshot of the *current* state, not the state at transfer time → leakage risk): `contract_expiration_date`, `market_value_in_eur`, `highest_market_value_in_eur`, `international_caps`, `international_goals`, `current_club_*`, `last_season`
- Not useful: names parts, `player_code`, `agent_name`, `image_url`, `url`, place of birth, `current_national_team_id`

**`player_valuations`**
- `player_id`, `date`, `market_value_in_eur` – last valuation before the transfer + its change over the previous 12 months

**`appearances`**
- `player_id`, `date`, `competition_id`, `goals`, `assists`, `minutes_played`, `yellow_cards`, `red_cards` – aggregated over the 12 months *before* the transfer date
- Dropped: `appearance_id`, `game_id`, `player_name`, `player_club_id`, `player_current_club_id`

**`clubs`**
- `club_id`, `name`, `domestic_competition_id`
- `total_market_value` is **100 % missing** → dropped
- The remaining numeric columns (`squad_size`, `average_age`, `foreigners_percentage`, ...) are current snapshots; whether to use them is open (see decisions)

**`competitions`**
- `competition_id`, `name`, `type`, `sub_type`, `country_name`, `confederation`
- There is **no** ready-made "major league" flag in this version of the data; league level will be derived from `competition_id` / country.

## 2. Cleaning

### 2.1 Column selection, date parsing, exact duplicates (no decision needed)

In [5]:
KEEP = {
    "transfers": ["player_id", "transfer_date", "transfer_season", "from_club_id", "to_club_id",
                  "from_club_name", "to_club_name", "player_name", "transfer_fee", "market_value_in_eur"],
    "players": ["player_id", "name", "date_of_birth", "position", "sub_position", "foot",
                "height_in_cm", "country_of_citizenship"],
    "player_valuations": ["player_id", "date", "market_value_in_eur"],
    "appearances": ["player_id", "date", "competition_id", "goals", "assists",
                    "minutes_played", "yellow_cards", "red_cards"],
    "clubs": ["club_id", "name", "domestic_competition_id"],
    "competitions": ["competition_id", "name", "type", "sub_type", "country_name", "confederation"],
}

transfers         = transfers[KEEP["transfers"]].copy()
players           = players[KEEP["players"]].copy()
player_valuations = player_valuations[KEEP["player_valuations"]].copy()
appearances       = appearances[KEEP["appearances"]].copy()
clubs             = clubs[KEEP["clubs"]].copy()
competitions      = competitions[KEEP["competitions"]].copy()

# dates -> datetime (unparseable values become NaT and are reported)
transfers["transfer_date"]      = pd.to_datetime(transfers["transfer_date"], errors="coerce")
players["date_of_birth"]        = pd.to_datetime(players["date_of_birth"], errors="coerce")
player_valuations["date"]       = pd.to_datetime(player_valuations["date"], errors="coerce")
appearances["date"]             = pd.to_datetime(appearances["date"], errors="coerce")

print("NaT after parsing:",
      transfers["transfer_date"].isna().sum(), players["date_of_birth"].isna().sum(),
      player_valuations["date"].isna().sum(), appearances["date"].isna().sum())

# exact duplicate transfers are safe to remove
before = len(transfers)
transfers = transfers.drop_duplicates()
print(f"transfers: removed {before - len(transfers)} exact duplicate rows")

NaT after parsing: 0 49 0 0
transfers: removed 0 exact duplicate rows


### 2.2 Missing values that have a single sensible treatment

- **`transfer_fee` missing** → the row is dropped. The target cannot be imputed (an imputed target would be invented data), and the model cannot learn from a row without a label.
- **`date_of_birth` missing** (players) → the row is dropped. Age at transfer is a core feature and cannot be reconstructed.
- **`market_value_in_eur` missing** (transfers) → the row is dropped (**decision of the author**: no recovery from `player_valuations`, only rows with a Transfermarkt market value at transfer time are modelled).

In [6]:
transfers_all = transfers.copy()   # kept only for the diagnostics of decision A

n0 = len(transfers)
transfers = transfers.dropna(subset=["transfer_fee"]).copy()
print(f"transfers: dropped {n0 - len(transfers)} rows without transfer_fee -> {len(transfers)} left")

n0 = len(transfers)
transfers = transfers.dropna(subset=["market_value_in_eur"]).copy()
print(f"transfers: dropped {n0 - len(transfers)} rows without market_value_in_eur -> {len(transfers)} left")

n0 = len(players)
players = players.dropna(subset=["date_of_birth"]).copy()
print(f"players: dropped {n0 - len(players)} rows without date_of_birth -> {len(players)} left")

transfers: dropped 61526 rows without transfer_fee -> 113639 left
transfers: dropped 26502 rows without market_value_in_eur -> 87137 left
players: dropped 49 rows without date_of_birth -> 50100 left


### 2.3 Open decisions (diagnostics only, nothing is changed here)

**PENDING DECISION A – transfers with `transfer_fee == 0`**

What the documentation says: in this dataset a fee of 0 stands for free transfers and `NaN` for a fee that was not given. The `transfers` table has **no column** that separates a real free transfer (expired contract) from a loan or an end of loan, so `0` may mix these cases (not confirmed by the official description). The cells below estimate it with a proxy: a transfer is *loan-like* if the player moves back to the club he left (or is moved back later) within ~3 years.

In [7]:
fee = transfers["transfer_fee"]
print("fee == 0:", int((fee == 0).sum()), "| fee > 0:", int((fee > 0).sum()))
display(fee[fee > 0].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(0))

# inspect what the zero-fee transfers look like
display(transfers[fee == 0].sample(10, random_state=42)[
    ["transfer_date", "player_name", "from_club_name", "to_club_name", "market_value_in_eur"]])

fee == 0: 70695 | fee > 0: 16442


count        16442.0
mean       5054869.0
std       10122405.0
min            500.0
5%          100000.0
25%         500000.0
50%        1500000.0
75%        5000000.0
95%       22000000.0
99%       50000000.0
max      222000000.0
Name: transfer_fee, dtype: float64

,transfer_date,player_name,from_club_name,to_club_name,market_value_in_eur
43311,2023-09-01,Slobodan Tedic,Man City,Charlton,600000.0
70817,2022-02-05,Sergiy Myakushko,FC Minaj,Kolos Kovalivka,150000.0
72733,2022-01-09,Satoki Uejo,Fagiano Okayama,Cerezo Osaka,600000.0
140265,2016-06-30,Mërgim Vojvoda,Carl Zeiss Jena,Standard U21,75000.0
100655,2020-01-01,Svetoslav Kovachev,Ludogorets,Arda Kardzhali,1000000.0
60070,2022-08-31,Vladyslav Vakula,Polissya,Shakhtar D.,300000.0
164631,2011-07-01,Kevin van Veen,Helmond Sport,Dijkse Boys,75000.0
62164,2022-07-22,Gaëtan Robail,Lens,Atromitos,900000.0
6132,2026-01-20,Justin Bourgault,Stade Brestois,AS Nancy,600000.0
140328,2016-06-30,Joel Fameyeh,Belshina,Wa All Stars,150000.0


In [8]:
# proxy for loan-like moves, computed on ALL transfers (before dropping rows)
ta = transfers_all.sort_values(["player_id", "transfer_date"]).copy()
g = ta.groupby("player_id")
prev_back = ((ta["from_club_id"] == g["to_club_id"].shift()) & (ta["to_club_id"] == g["from_club_id"].shift())
             & ((ta["transfer_date"] - g["transfer_date"].shift()).dt.days <= 1100))
next_back = ((ta["from_club_id"] == g["to_club_id"].shift(-1)) & (ta["to_club_id"] == g["from_club_id"].shift(-1))
             & ((g["transfer_date"].shift(-1) - ta["transfer_date"]).dt.days <= 1100))
ta["loan_like"] = prev_back | next_back
ta["fee_group"] = np.select([ta["transfer_fee"].isna(), ta["transfer_fee"] == 0], ["NaN", "0"], "> 0")

display(ta.groupby("fee_group")["loan_like"].agg(rows="size", share_loan_like="mean").round(3))

,rows,share_loan_like
fee_group,,
0,96085,0.636
> 0,17554,0.172
NaN,61526,0.052


**PENDING DECISION B – missing profile attributes of players** (`height_in_cm`, `foot`, `sub_position`, `country_of_citizenship`)

Only players who actually appear in the (fee-known) transfers matter.

In [9]:
involved = players[players["player_id"].isin(transfers["player_id"])]
print("players with a known-fee transfer:", len(involved))
display(missing_report(involved))

print("transfers whose player is not in `players` (after cleaning):",
      int((~transfers["player_id"].isin(players["player_id"])).sum()))

players with a known-fee transfer: 17907


,missing,missing_%,dtype
foot,799,4.46,object
height_in_cm,367,2.05,float64
sub_position,9,0.05,object


transfers whose player is not in `players` (after cleaning): 10


**Other items to look at when merging (not blocking):**
- `competitions.country_name` is missing for international competitions – check which ones:

In [10]:
display(competitions[competitions["country_name"].isna()][["competition_id", "name", "type"]])

,competition_id,name,type
1,AFAC,afc-asian-cup,national_team_competition
2,AFCN,africa-cup-of-nations,national_team_competition
12,CL,uefa-champions-league,international_cup
13,CLQ,uefa-champions-league-qualifying,international_cup
14,COPA,copa-america,national_team_competition
19,ECLQ,uefa-conference-league-qualifying,other
21,EL,uefa-europa-league,other
22,ELQ,uefa-europa-league-qualifying,other
24,EURO,uefa-euro,national_team_competition
26,FIWC,world-cup,national_team_competition


## Status

- [x] Datasets described, attributes selected
- [x] Columns selected, dates parsed, duplicates removed, rows without target / date of birth dropped
- [ ] Decision A: zero-fee transfers
- [x] Rows without `market_value_in_eur` dropped (author's decision)
- [ ] Decision B: missing `height_in_cm`, `foot`, `sub_position`, `country_of_citizenship`
- [ ] Save cleaned tables to `data/processed/`